# Candidate list comparison

Compare peptide sequences from a **generated** `final_candidates.csv` run against the original paper PDDP table (`Candidates Table 6 - Database of PDDPs.csv`).

Reports:

- **Exact matches** — paper sequence appears verbatim in the generated set
- **Near matches** — same core sequence with a few extra or missing residues (substring containment within a configurable length delta)

Edit paths in the first code cell, then run all cells. The final cell runs a small test suite and prints a pass/fail summary.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("../sequence_to_svm_minimal").expanduser().resolve()

# --- edit these paths (relative to PROJECT_ROOT unless absolute) ---
GENERATED_CSV = "data/proteomes/paper_pddp_relaxed/keep_dup/final_candidates.csv"
PAPER_CSV = "data/proteomes/original_paper/Candidates Table 6 - Database of PDDPs.csv"

GENERATED_SEQUENCE_COL = "sequence"
PAPER_SEQUENCE_COL = "Sequence"

# Max |len(paper) - len(generated)| for a near match (substring containment).
MAX_NEAR_MATCH_DELTA = 3

# Optional minimum overlap rates for informational assertions (set None to skip).
MIN_EXACT_MATCH_RATE: float | None = None
MIN_COMBINED_COVERAGE_RATE: float | None = None


def _resolve(path_str: str) -> Path:
    p = Path(path_str).expanduser()
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    return p.resolve()


def normalize_sequence(value) -> str:
    return str(value).strip().upper()


def load_generated_sequences(path: Path, column: str = GENERATED_SEQUENCE_COL) -> pd.Series:
    df = pd.read_csv(path, usecols=[column])
    return df[column].map(normalize_sequence)


def load_paper_table(path: Path, column: str = PAPER_SEQUENCE_COL) -> pd.DataFrame:
    df = pd.read_csv(path)
    if column not in df.columns:
        raise KeyError(f"Paper CSV missing column {column!r}; found {list(df.columns)}")
    out = df.copy()
    out[column] = out[column].map(normalize_sequence)
    return out


def index_sequences_by_length(sequences: pd.Series) -> dict[int, list[str]]:
    by_length: dict[int, list[str]] = {}
    for seq in sequences.drop_duplicates():
        by_length.setdefault(len(seq), []).append(seq)
    return by_length


@dataclass(frozen=True)
class NearMatch:
    paper_sequence: str
    generated_sequence: str
    length_delta: int
    match_type: str  # "paper_in_generated" | "generated_in_paper"

    @property
    def label(self) -> str:
        return {
            "paper_in_generated": "paper ⊆ generated (extension)",
            "generated_in_paper": "generated ⊆ paper (truncation)",
        }[self.match_type]


def find_near_matches(
    paper_sequence: str,
    generated_by_length: dict[int, list[str]],
    *,
    max_delta: int = MAX_NEAR_MATCH_DELTA,
) -> list[NearMatch]:
    paper = normalize_sequence(paper_sequence)
    paper_len = len(paper)
    matches: list[NearMatch] = []

    for delta in range(1, max_delta + 1):
        for generated_len in (paper_len + delta, paper_len - delta):
            if generated_len <= 0:
                continue
            for generated in generated_by_length.get(generated_len, []):
                if paper in generated:
                    matches.append(
                        NearMatch(paper, generated, delta, "paper_in_generated")
                    )
                elif generated in paper:
                    matches.append(
                        NearMatch(paper, generated, delta, "generated_in_paper")
                    )
    return matches


def best_near_match(
    paper_sequence: str,
    generated_by_length: dict[int, list[str]],
    *,
    max_delta: int = MAX_NEAR_MATCH_DELTA,
) -> NearMatch | None:
    matches = find_near_matches(
        paper_sequence, generated_by_length, max_delta=max_delta
    )
    if not matches:
        return None
    return min(matches, key=lambda m: (m.length_delta, len(m.generated_sequence)))


def build_comparison_table(
    paper_df: pd.DataFrame,
    generated_sequences: pd.Series,
    *,
    paper_col: str = PAPER_SEQUENCE_COL,
    max_delta: int = MAX_NEAR_MATCH_DELTA,
) -> pd.DataFrame:
    generated_set = set(generated_sequences)
    generated_by_length = index_sequences_by_length(generated_sequences)

    rows: list[dict] = []
    for paper_seq, group in paper_df.groupby(paper_col, sort=False):
        exact = paper_seq in generated_set
        near = None if exact else best_near_match(
            paper_seq, generated_by_length, max_delta=max_delta
        )

        if exact:
            status = "exact"
        elif near is not None:
            status = "near"
        else:
            status = "missing"

        rows.append(
            {
                "paper_sequence": paper_seq,
                "paper_rows": len(group),
                "paper_length": len(paper_seq),
                "status": status,
                "match_type": "exact" if exact else (near.match_type if near else None),
                "match_label": "exact" if exact else (near.label if near else None),
                "generated_sequence": paper_seq if exact else (near.generated_sequence if near else None),
                "length_delta": 0 if exact else (near.length_delta if near else None),
                "gene_names": ";".join(
                    sorted({str(v) for v in group.get("Gene names", pd.Series(dtype=str)).dropna() if str(v)})
                )
                if "Gene names" in group.columns
                else None,
            }
        )

    return pd.DataFrame(rows)


GENERATED_PATH = _resolve(GENERATED_CSV)
PAPER_PATH = _resolve(PAPER_CSV)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"GENERATED_CSV: {GENERATED_PATH}")
print(f"PAPER_CSV: {PAPER_PATH}")

PROJECT_ROOT: C:\Users\bioin\Documents\Peptide-Anti-microbial-Properties-Prediction\sequence_to_svm_minimal
GENERATED_CSV: C:\Users\bioin\Documents\Peptide-Anti-microbial-Properties-Prediction\sequence_to_svm_minimal\data\proteomes\paper_pddp_relaxed\keep_dup\final_candidates.csv
PAPER_CSV: C:\Users\bioin\Documents\Peptide-Anti-microbial-Properties-Prediction\sequence_to_svm_minimal\data\proteomes\original_paper\Candidates Table 6 - Database of PDDPs.csv


In [2]:
paper_df = load_paper_table(PAPER_PATH)
generated_sequences = load_generated_sequences(GENERATED_PATH)

comparison_df = build_comparison_table(
    paper_df,
    generated_sequences,
    max_delta=MAX_NEAR_MATCH_DELTA,
)

n_paper_rows = len(paper_df)
n_paper_unique = paper_df[PAPER_SEQUENCE_COL].nunique()
n_generated_unique = generated_sequences.nunique()

status_counts = comparison_df["status"].value_counts()
exact_count = int(status_counts.get("exact", 0))
near_count = int(status_counts.get("near", 0))
missing_count = int(status_counts.get("missing", 0))
covered_count = exact_count + near_count

print("Dataset sizes")
print(f"  paper rows:              {n_paper_rows:,}")
print(f"  paper unique sequences:  {n_paper_unique:,}")
print(f"  generated unique seqs:   {n_generated_unique:,}")
print()
print("Unique paper sequence coverage in generated set")
print(f"  exact matches:           {exact_count:,} ({exact_count / n_paper_unique:.1%})")
print(f"  near matches only:       {near_count:,} ({near_count / n_paper_unique:.1%})")
print(f"  combined coverage:       {covered_count:,} ({covered_count / n_paper_unique:.1%})")
print(f"  missing:                 {missing_count:,} ({missing_count / n_paper_unique:.1%})")
print()
print("Near-match breakdown")
print(comparison_df.loc[comparison_df["status"] == "near", "match_label"].value_counts().to_string())

Dataset sizes
  paper rows:              1,074
  paper unique sequences:  1,045
  generated unique seqs:   44,455

Unique paper sequence coverage in generated set
  exact matches:           956 (91.5%)
  near matches only:       18 (1.7%)
  combined coverage:       974 (93.2%)
  missing:                 71 (6.8%)

Near-match breakdown
match_label
generated ⊆ paper (truncation)    11
paper ⊆ generated (extension)      7


In [3]:
from IPython.display import display

display_cols = [
    "paper_sequence",
    "paper_length",
    "status",
    "match_label",
    "generated_sequence",
    "length_delta",
    "gene_names",
]

print("Exact matches (sample)")
display(comparison_df.loc[comparison_df["status"] == "exact", display_cols].head(15))

print("Near matches (sample)")
display(comparison_df.loc[comparison_df["status"] == "near", display_cols].head(15))

print("Missing from generated set (sample)")
display(comparison_df.loc[comparison_df["status"] == "missing", display_cols].head(15))

Exact matches (sample)


,paper_sequence,paper_length,status,match_label,generated_sequence,length_delta,gene_names
0,LSLVTKKKRFWCWQRPKYQFL,21,exact,exact,LSLVTKKKRFWCWQRPKYQFL,0.0,DFNA5
1,RMFRGSLYKRYPSLWRRL,18,exact,exact,RMFRGSLYKRYPSLWRRL,0.0,SMARCB1
2,SLVTKKKRFWCWQRPKYQFL,20,exact,exact,SLVTKKKRFWCWQRPKYQFL,0.0,DFNA5
3,VTKKKRFWCWQRPKYQFL,18,exact,exact,VTKKKRFWCWQRPKYQFL,0.0,DFNA5
4,SQLYPRHKHLLIKRSLRCRKC,21,exact,exact,SQLYPRHKHLLIKRSLRCRKC,0.0,DCTN4
6,RKLAVNMVPFPRLHFFMPGFAPLTSRGSQQYR,32,exact,exact,RKLAVNMVPFPRLHFFMPGFAPLTSRGSQQYR,0.0,TUBB4A;TUBB;TUBB4B;TUBB2B;TUBB2A;TUBB6;TUBB8
8,SQLYPRHKHLLIKRSLRCR,19,exact,exact,SQLYPRHKHLLIKRSLRCR,0.0,DCTN4
9,RKLAVNMVPFPRLHFFMPGFAPLTSR,26,exact,exact,RKLAVNMVPFPRLHFFMPGFAPLTSR,0.0,TUBB4A;TUBB;TUBB4B;TUBB2B;TUBB2A;TUBB6;TUBB8
10,RSIFIKYKSKPFCEKLLSWVKS,22,exact,exact,RSIFIKYKSKPFCEKLLSWVKS,0.0,PSMG2
11,SRVPRFYWKDRLLKMKMA,18,exact,exact,SRVPRFYWKDRLLKMKMA,0.0,GLB1


Near matches (sample)


,paper_sequence,paper_length,status,match_label,generated_sequence,length_delta,gene_names
42,KKYLKKNNLRDWLRVV,16,near,generated ⊆ paper (truncation),KYLKKNNLRDWLRVV,1.0,RPL22;RPL22L1
249,AWKISGFPKNRVIGSGCNLDSARFRYLMG,29,near,generated ⊆ paper (truncation),AWKISGFPKNRVIGSGCNLDSARFRYL,2.0,LDHA
320,TKTVKKAARVIIEKYYTRL,19,near,generated ⊆ paper (truncation),TKTVKKAARVIIEKYYT,2.0,RPS17
336,KSGKYVLGYKQTLKMIR,17,near,paper ⊆ generated (extension),KSGKYVLGYKQTLKMIRQ,1.0,RPL30
388,GKVKVGVNGFGRIGRLVTR,19,near,paper ⊆ generated (extension),MGKVKVGVNGFGRIGRLVTR,1.0,GAPDH
394,GKVKVGVNGFGRIGRLVTRA,20,near,paper ⊆ generated (extension),MGKVKVGVNGFGRIGRLVTRA,1.0,GAPDH
441,FAKRNVKLIAL,11,near,paper ⊆ generated (extension),APEFAKRNVKLIAL,3.0,PRDX6
469,YTRAASTARHLYLRGGAGVGSMTKIYGGRQR,31,near,generated ⊆ paper (truncation),TRAASTARHLYLRGGAGVGSMTKIYGGRQR,1.0,RPS19
495,VFRRFVEVGRVAYVSFGPHAGKLV,24,near,generated ⊆ paper (truncation),RRFVEVGRVAYVSFGPHAGKLV,2.0,RPL14
605,NRIGKVGNQKRVVGVLLGSW,20,near,generated ⊆ paper (truncation),NRIGKVGNQKRVVGVLLG,2.0,PSMD7


Missing from generated set (sample)


,paper_sequence,paper_length,status,match_label,generated_sequence,length_delta,gene_names
5,MRAKWRKKRMRRLK,14,missing,NaN,NaN,NaN,RPL41
7,GFVKVVKNKAYFKRYQVKF,19,missing,NaN,NaN,NaN,RPL5
16,GHQQLYWSHPRKFGQGSRSCRVCSNRHGLIRKYGLNMC,38,missing,NaN,NaN,NaN,RPS29
17,GHQQLYWSHPRKFGQGSRSCRVCSNRHGLIRKYGLNM,37,missing,NaN,NaN,NaN,RPS29
23,GHQQLYWSHPRKFGQGSRSCRVCSNRHGLIRKYGL,35,missing,NaN,NaN,NaN,RPS29
49,GHQQLYWSHPRKFGQGSRSCRVCSNRHGLIRKY,33,missing,NaN,NaN,NaN,RPS29
90,VQSVISLIMGMKFFRVKMYP,20,missing,NaN,NaN,NaN,FRYL
114,AARRALHFVFKVGNRF,16,missing,NaN,NaN,NaN,GLOD4
116,MAAVARAVTLMTPLPFLLRR,20,missing,NaN,NaN,NaN,OR52K1
120,NWLLQRPGQSPKRLIYLVSK,20,missing,NaN,NaN,NaN,light_kappa


In [4]:
@dataclass
class TestResult:
    name: str
    passed: bool
    detail: str


def _record(results: list[TestResult], name: str, passed: bool, detail: str = "") -> None:
    results.append(TestResult(name=name, passed=passed, detail=detail))


def run_candidate_list_tests(
    paper_df: pd.DataFrame,
    generated_sequences: pd.Series,
    comparison_df: pd.DataFrame,
    *,
    paper_col: str = PAPER_SEQUENCE_COL,
    max_delta: int = MAX_NEAR_MATCH_DELTA,
    min_exact_match_rate: float | None = MIN_EXACT_MATCH_RATE,
    min_combined_coverage_rate: float | None = MIN_COMBINED_COVERAGE_RATE,
) -> pd.DataFrame:
    results: list[TestResult] = []

    _record(
        results,
        "paper_csv_has_sequence_column",
        paper_col in paper_df.columns,
        f"expected column {paper_col!r}",
    )
    _record(
        results,
        "paper_table_non_empty",
        len(paper_df) > 0,
        f"rows={len(paper_df)}",
    )
    _record(
        results,
        "generated_set_non_empty",
        generated_sequences.nunique() > 0,
        f"unique={generated_sequences.nunique()}",
    )
    _record(
        results,
        "comparison_has_one_row_per_unique_paper_sequence",
        len(comparison_df) == paper_df[paper_col].nunique(),
        f"comparison={len(comparison_df)}, unique paper={paper_df[paper_col].nunique()}",
    )

    statuses = set(comparison_df["status"].dropna().unique())
    _record(
        results,
        "comparison_status_values_valid",
        statuses.issubset({"exact", "near", "missing"}),
        f"found={sorted(statuses)}",
    )

    exact_rows = comparison_df.loc[comparison_df["status"] == "exact"]
    generated_set = set(generated_sequences)
    exact_in_generated = exact_rows["paper_sequence"].isin(generated_set).all()
    _record(
        results,
        "exact_matches_present_in_generated_set",
        exact_in_generated,
        f"exact rows={len(exact_rows)}",
    )

    near_rows = comparison_df.loc[comparison_df["status"] == "near"]
    near_delta_ok = near_rows["length_delta"].between(1, max_delta).all()
    _record(
        results,
        "near_matches_respect_length_delta",
        bool(near_delta_ok) if len(near_rows) else True,
        f"near rows={len(near_rows)}, max_delta={max_delta}",
    )

    near_disjoint_from_exact = not near_rows["paper_sequence"].isin(exact_rows["paper_sequence"]).any()
    _record(
        results,
        "near_matches_disjoint_from_exact",
        near_disjoint_from_exact,
        "",
    )

    def _substring_holds(row) -> bool:
        paper = row["paper_sequence"]
        generated = row["generated_sequence"]
        if row["match_type"] == "paper_in_generated":
            return paper in generated
        if row["match_type"] == "generated_in_paper":
            return generated in paper
        return False

    if len(near_rows):
        substring_ok = near_rows.apply(_substring_holds, axis=1).all()
    else:
        substring_ok = True
    _record(
        results,
        "near_matches_are_substring_pairs",
        bool(substring_ok),
        f"near rows={len(near_rows)}",
    )

    n_unique = paper_df[paper_col].nunique()
    exact_rate = len(exact_rows) / n_unique
    combined_rate = (
        comparison_df["status"].isin(["exact", "near"]).sum() / n_unique
    )

    _record(
        results,
        "report_exact_match_rate",
        True,
        f"{exact_rate:.1%} ({len(exact_rows)}/{n_unique} unique paper sequences)",
    )
    _record(
        results,
        "report_combined_coverage_rate",
        True,
        f"{combined_rate:.1%} ({comparison_df['status'].isin(['exact', 'near']).sum()}/{n_unique})",
    )

    if min_exact_match_rate is not None:
        _record(
            results,
            "exact_match_rate_meets_threshold",
            exact_rate >= min_exact_match_rate,
            f"rate={exact_rate:.1%}, threshold={min_exact_match_rate:.1%}",
        )
    if min_combined_coverage_rate is not None:
        _record(
            results,
            "combined_coverage_meets_threshold",
            combined_rate >= min_combined_coverage_rate,
            f"rate={combined_rate:.1%}, threshold={min_combined_coverage_rate:.1%}",
        )

    return pd.DataFrame([r.__dict__ for r in results])


test_results = run_candidate_list_tests(
    paper_df,
    generated_sequences,
    comparison_df,
    max_delta=MAX_NEAR_MATCH_DELTA,
)

n_pass = int(test_results["passed"].sum())
n_fail = int((~test_results["passed"]).sum())

print(f"Tests: {n_pass} passed, {n_fail} failed, {len(test_results)} total")
display(test_results)

if n_fail:
    raise AssertionError(f"{n_fail} candidate list comparison test(s) failed")

Tests: 11 passed, 0 failed, 11 total


,name,passed,detail
0,paper_csv_has_sequence_column,True,expected column 'Sequence'
1,paper_table_non_empty,True,rows=1074
2,generated_set_non_empty,True,unique=44455
3,comparison_has_one_row_per_unique_paper_sequence,True,"comparison=1045, unique paper=1045"
4,comparison_status_values_valid,True,"found=['exact', 'missing', 'near']"
5,exact_matches_present_in_generated_set,True,exact rows=956
6,near_matches_respect_length_delta,True,"near rows=18, max_delta=3"
7,near_matches_disjoint_from_exact,True,
8,near_matches_are_substring_pairs,True,near rows=18
9,report_exact_match_rate,True,91.5% (956/1045 unique paper sequences)
